# IOAI — 2025 Contest Broken Bert (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, glob, zipfile
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/val_dataset.csv'):
    os.environ['KAGGLE_API_TOKEN'] = 'KGAT_5dd261fded8e0d7eb2c29d8d65dfabea'  # 내장 토큰(규칙 수락된 계정)
    os.system('pip -q install kaggle')
    from kaggle.api.kaggle_api_extended import KaggleApi
    api = KaggleApi(); api.authenticate()
    api.competition_download_files('neoai-2025-broken-bert', path='data', quiet=False)
    for zp in glob.glob('data/*.zip'):
        with zipfile.ZipFile(zp) as zf: zf.extractall('data')
        os.remove(zp)
print('데이터 준비:', 'data/val_dataset.csv' if os.path.exists('data/val_dataset.csv') else '실패')
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Broken BERT · 모범답안 (임베딩 복구)

**통찰**: 손상된 건 **임베딩 행렬의 일부 행**뿐 — 분류 헤드·트랜스포머 층은 멀쩡합니다. broken_bert 는
`bert-base-uncased` 를 미세조정한 모델이고 **vocab(토큰 id)이 동일**하므로, **0이 된 행만 건강한
bert-base-uncased 임베딩으로 채우면**(검증셋 학습 안 함 = 누수 없음) 성능이 크게 회복됩니다.
검증셋 macro-F1: 손상 ≈0.29 → **도너 복구 ≈0.60**.

> 더 가벼운 대안(레퍼런스): 손상 토큰을 이루는 *서브토큰들의 임베딩 평균*으로 채우기(≈0.43). 아래 마지막 셀 주석 참고.

In [ ]:
import torch, pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModel
from sklearn.metrics import f1_score
dev = 'cuda' if torch.cuda.is_available() else 'cpu'; print('device', dev)
tok = AutoTokenizer.from_pretrained('Ilseyar-kfu/broken_bert')
model = AutoModelForSequenceClassification.from_pretrained('Ilseyar-kfu/broken_bert').to(dev).eval()
val = pd.read_csv('data/val_dataset.csv')


In [ ]:
# 0이 된 임베딩 행을 건강한 bert-base-uncased(동일 vocab) 임베딩으로 복구
emb = model.bert.embeddings.word_embeddings.weight.detach().cpu()
zmask = (emb == 0).all(dim=1)
print('복구할 0 임베딩 행:', int(zmask.sum()))
donor = AutoModel.from_pretrained('bert-base-uncased').embeddings.word_embeddings.weight.detach().cpu()
emb[zmask] = donor[zmask]                                   # 같은 토큰 id 의 건강한 벡터로 채움
model.bert.embeddings.word_embeddings.weight.data.copy_(emb.to(dev))


In [ ]:
@torch.no_grad()
def predict_labels(model, texts, bs=64):
    ID2LAB = {0: 'neutral', 1: 'positive', 2: 'negative'}   # 대회 고정 매핑
    out = []
    for i in range(0, len(texts), bs):
        e = tok(texts[i:i+bs], truncation=True, padding=True, max_length=128, return_tensors='pt').to(dev)
        out += [ID2LAB[j] for j in model(**e).logits.argmax(-1).cpu().tolist()]
    return out

In [ ]:
pred = predict_labels(model, val['text'].tolist())
print('held-out macro-F1:', round(f1_score(val['labels'], pred, average='macro'), 4))
pd.DataFrame({'id': val['id'], 'labels': pred}).to_csv('submission.csv', index=False)
print('wrote submission.csv')

# --- 더 가벼운 대안(레퍼런스, ≈0.43): 서브토큰 평균 복구 ---
# i2t = {v:k for k,v in tok.get_vocab().items()}; zidx = torch.nonzero(zmask).squeeze().tolist()
# nz = {v:k for k,v in tok.get_vocab().items() if not zmask[v]}
# for zi in zidx:
#     subs = [idx for idx,st in nz.items() if st in i2t[zi]]
#     if subs: emb[zi] = emb[subs].mean(0)


## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)